### I am taking the information from 01_eda.ipynb and creating a pipeline in this notebook

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, KBinsDiscretizer, StandardScaler
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score
import joblib

In [4]:
df = pd.read_csv("../data/raw/labeled_insurance.csv")

print(df.head(10))

   age  tenure vehicle_type  claims_history  claims_count
0   56      10        Sedan               0             0
1   69      16        Sedan               0             0
2   46      12        Truck               0             0
3   32       0          SUV               1             1
4   60       1        Truck               1             1
5   25       8        Truck               0             0
6   78       2          SUV               0             0
7   38       0   Motorcycle               0             0
8   56      15        Sedan               0             1
9   75       5   Motorcycle               1             2


In [8]:
X = df.drop("claims_count", axis=1)
y = df["claims_count"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.head(5))
print() 
print(y_train.head(5))

     age  tenure vehicle_type  claims_history
29    38       0          SUV               0
535   31       1        Truck               0
695   68      11          SUV               0
557   59       1        Truck               2
836   63      16          SUV               1

29     1
535    0
695    0
557    2
836    1
Name: claims_count, dtype: int64


In [ ]:
numeric_features = ["age", "tenure ", "claims_history"]  
categorical_features = ["vehicle_type"]  

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        # Bucket only age variable
        ("age_bucket", KBinsDiscretizer(n_bins=4, encode="onehot-dense", strategy="uniform"), ["age"]),
        # Scale other numeric variable
        ("num_scale", StandardScaler(), ["tenure", "claims_history"]),
        # One-hot encode categorical variable
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["vehicle_type"])
    ]
)


In [ ]:
pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42))
    ]
)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
joblib.dump(pipeline, "insurance_pipeline.joblib")